In [12]:
import numpy as np
import pandas as pd
import os

In [13]:
NUM_ROWS = 5000
OUTPUT_DIR = "data"
OUTPUT_FILE = "pricing_data.csv"
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [14]:
#multipliers
df = pd.DataFrame({
    'duration_hours':      np.random.randint(1, 6, size=NUM_ROWS),
    'lead_time_hours':     np.random.uniform(0.5, 168, size=NUM_ROWS),
    'zone_center':         np.random.choice([0, 1], size=NUM_ROWS),
    'special_reqs_count':  np.random.randint(0, 3, size=NUM_ROWS),
    'demand_supply_ratio': np.random.uniform(0.5, 3.0, size=NUM_ROWS)
})

In [15]:
services = ["General House Cleaning",
      "Home Organizing",
      "Deep Cleaning",
      "Aircond Cleaning",
      "Carpet Cleaning",
      "Post-Renovation Cleaning",
      "Sofa or Mattress Cleaning",  
     "Plumbing Services",
      "Air Conditioner Repair",
      "Electrical Repair",
      "Washing Machine Repair",
      "Refrigerator Repair",
      "Door & Lock Repair",
      "Ceiling Repair", 
     "Furniture Assembly",
      "Mounting",
      "Painting & Touch-up Work",
      "Curtain or Blind Installation",
      "Minor Welding Jobs",
      "Kitchen Remodeling",
      "Tiling & Flooring",
      "Electrical Safety Check",
      "Gas Leak Detection",
      "Fire Extinguisher Servicing", 
     "House Moving",
      "Large Item Delivery",
      "Small Item Delivery", 
     "Lawn Mowing",
      "Gardening",
      "Tree Cutting",
      "Roof or Gutter Cleaning"
    ]

#base rate per hour or per unit
base_rates = {
    'general_house_cleaning': 15,
    'home_organizing': 20,
    'deep_cleaning': 25,
    'aircond_cleaning': 50,
    'carpet_cleaning': 30,
    'post_renovation_cleaning': 60,
    'sofa_or_mattress_cleaning': 30,
    'plumbing_services': 25,
    'air_conditioner_repair': 30,
    'electrical_repair': 25,
    'washing_machine_repair': 30,
    'refrigerator_repair': 35,
    'door_lock_repair': 20,
    'ceiling_repair': 25,
    'furniture_assembly': 20,
    'mounting': 20,
    'painting_touch_up_work': 25,
    'curtain_or_blind_installation': 20,
    'minor_welding_jobs': 30,
    'kitchen_remodeling': 80,
    'tiling_flooring': 35,
    'electrical_safety_check': 30,
    'gas_leak_detection': 40,
    'fire_extinguisher_servicing': 35,
    'house_moving': 25,
    'large_item_delivery': 30,
    'small_item_delivery': 20,
    'lawn_mowing': 25,
    'gardening': 20,
    'tree_cutting': 30,
    'roof_or_gutter_cleaning': 40
}

service_choices = np.random.choice(services, size=NUM_ROWS)
df['service'] = service_choices
dummies = pd.get_dummies(df['service'], prefix='service')

# Replace all spaces to underscore and lowercase all letters
dummies.columns = (
    dummies.columns
           .str.lower()
           .str.replace(r'[^0-9a-z_]+','_', regex=True)
           .str.rstrip('_')
)

# One-hot encoding
df = pd.concat([df, dummies], axis=1)

# Compute base rate
def get_base_rate(row):
    for col in dummies.columns:
        if row[col] == 1:
            key = col.replace('service_', '')
            return base_rates.get(key, 0)
    return 0

df['base_rate'] = df.apply(get_base_rate, axis=1)

# Price calculation
def rule_based_price(row):
    price = row['base_rate'] * row['duration_hours']
    price *= 0.9 if row['lead_time_hours'] > 24 else 1.2
    price *= 1.1 if row['zone_center'] == 1 else 1.0
    price += 3 * row['special_reqs_count']
    price *= row['demand_supply_ratio']
    return round(price, 2)

df['price'] = df.apply(rule_based_price, axis=1)



In [16]:
# Save and preview
output_path = os.path.join(OUTPUT_DIR, OUTPUT_FILE)
df.to_csv(output_path, index=False)
print(f"Pricing data saved to: {output_path}")
print(df[['service', 'base_rate', 'price']].head())


Pricing data saved to: data/pricing_data.csv
               service  base_rate   price
0      Home Organizing         20   45.54
1      Carpet Cleaning         30  229.10
2  Small Item Delivery         20   56.06
3  Large Item Delivery         30   52.28
4      Carpet Cleaning         30   62.19
